In [18]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../..')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config

client = config.create_minio_client()

[Bucket('aws'), Bucket('azure'), Bucket('google')]


In [19]:
object_name = "Compute_Engine/page_1.json"

try:
    response = client.get_object(config.PROVIDERS.get("google").get("bucket"), object_name=object_name)

    df = pd.read_json(response)

    response.close()
    response.release_conn()

    print ("Success. Page loaded and converted into dataframe")
    print ("Array size: Rows = ",df.shape[0], " and Columns = ", df.shape[1])

except Exception as e:
    print ("Error: ",e)


Success. Page loaded and converted into dataframe
Array size: Rows =  5000  and Columns =  2


In [20]:
#With the commnand bellow, we open the first level key : value pairs in columns, and the first level inner dicts also open, 
#in the form key.value (ex: category.serviceDisplayName, <- This was an inner dict category :{key:value, key:value})

df_flat = pd.json_normalize(df['skus'])
# df_flat.head(3)
df_flat[['skuId', 'category.serviceDisplayName', 'category.resourceFamily', 'category.usageType', 'category.resourceGroup']].head()

,skuId,category.serviceDisplayName,category.resourceFamily,category.usageType,category.resourceGroup
0,0001-B904-8A40,Compute Engine,Compute,OnDemand,CPU
1,0001-FC8F-A9AF,Compute Engine,Compute,Preemptible,CPU
2,0006-C9C8-BB6F,Compute Engine,Compute,Commit1Yr,CPU
3,0007-4724-5A32,Compute Engine,Compute,OnDemand,CPU
4,0007-9388-EF75,Compute Engine,Compute,OnDemand,RAM


In [21]:
#Here geoTaxonomy.regions looks like this: ["value"]
df_flat[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()

#We apply the explode in the geoTaxonomy.regions column and store the res in a new dataframe
df_flat2 = df_flat.explode('geoTaxonomy.regions')

#The result will be: geoTaxonomy wont be a list anymore, and all the list elements will be in a single line
# df_flat2[['skuId','geoTaxonomy.type', 'geoTaxonomy.regions']].head()
df_flat2.head(3)


,name,skuId,description,serviceRegions,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,[southamerica-west1],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,[europe-west9],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,[us-west8],"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8


In [22]:
#We apply the same proccedure as the cell above. This time we open the list serviceRegions
df_flat2[['skuId', 'serviceRegions']].head()

df_flat3 = df_flat2.explode('serviceRegions')

df_flat3[['skuId', 'serviceRegions', 'geoTaxonomy.regions']].head()

df_flat3.head()

,name,skuId,description,serviceRegions,pricingInfo,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,southamerica-west1,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,europe-west9,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,us-west8,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8
3,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,europe-north1,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1
4,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,northamerica-northeast2,"[{'summary': '', 'pricingExpression': {'usageU...",Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2


In [23]:
#The only column not fully opened yet is the pricingInfo: structure -> [{key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}}]
df_flat3[['skuId', 'pricingInfo']].head()

#This first explode removes the list: we have now -> {key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}} 
df_flat4 = df_flat3.explode('pricingInfo')
df_flat4[['skuId', 'pricingInfo']].head()
# df_flat4.head()

,skuId,pricingInfo
0,0001-B904-8A40,"{'summary': '', 'pricingExpression': {'usageUn..."
1,0001-FC8F-A9AF,"{'summary': '', 'pricingExpression': {'usageUn..."
2,0006-C9C8-BB6F,"{'summary': '', 'pricingExpression': {'usageUn..."
3,0007-4724-5A32,"{'summary': '', 'pricingExpression': {'usageUn..."
4,0007-9388-EF75,"{'summary': '', 'pricingExpression': {'usageUn..."


In [24]:
#We are here now: {key1:val1, key2:val2, key3:val3, key4 : {key:val, key:[{}]}} -> we can open the dict with the normalize
#A new dataframe will be created with the pricing info and then concatenated with the original dataframe

pricing1 = pd.json_normalize(df_flat4['pricingInfo'])
pricing1.head()

,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.tieredRates,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount
0,,1,2026-05-18T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
1,,1,2026-05-18T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
2,,1,2026-05-18T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
3,,1,2026-05-18T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
4,,1,2026-05-18T07:00:00Z,GBy.h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",gigabyte hour,By.s,byte second,3.600000e+12,NaN,NaN,NaN


In [25]:
#I now have to concatenate the 2 dataframes beeing carefull though with the indexes
pricing1.index = df_flat4.index

df_flat5 = pd.concat([df_flat4.drop(columns=['pricingInfo']), pricing1], axis=1)
df_flat5.head()

,name,skuId,description,serviceRegions,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.tieredRates,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,southamerica-west1,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,,1,2026-05-18T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,europe-west9,Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9,,1,2026-05-18T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,us-west8,Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8,,1,2026-05-18T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
3,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,europe-north1,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,,1,2026-05-18T07:00:00Z,h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",hour,s,second,3.600000e+03,NaN,NaN,NaN
4,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,northamerica-northeast2,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,,1,2026-05-18T07:00:00Z,GBy.h,1,"[{'startUsageAmount': 0, 'unitPrice': {'curren...",gigabyte hour,By.s,byte second,3.600000e+12,NaN,NaN,NaN


In [26]:
df_flat6 = df_flat5.explode('pricingExpression.tieredRates')

df_flat6[['skuId', 'pricingExpression.tieredRates']].head()

,skuId,pricingExpression.tieredRates
0,0001-B904-8A40,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
1,0001-FC8F-A9AF,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
2,0006-C9C8-BB6F,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
3,0007-4724-5A32,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."
4,0007-9388-EF75,"{'startUsageAmount': 0, 'unitPrice': {'currenc..."


In [27]:
pricing2 = pd.json_normalize(df_flat6['pricingExpression.tieredRates'])
pricing2.head()

,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos
0,0.0,USD,0,4954950.0
1,0.0,USD,0,11470000.0
2,0.0,USD,0,20550000.0
3,0.0,USD,0,3815312.0
4,0.0,USD,0,6243470.0


In [28]:
pricing2.index = df_flat6.index

df_final_flat = pd.concat([df_flat6.drop(columns=['pricingExpression.tieredRates']), pricing2], axis=1)
pd.set_option('display.max_columns', None)
df_final_flat.head()

,name,skuId,description,serviceRegions,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,southamerica-west1,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,,1,2026-05-18T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,4954950.0
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,europe-west9,Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9,,1,2026-05-18T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,11470000.0
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,us-west8,Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8,,1,2026-05-18T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,20550000.0
3,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,europe-north1,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,,1,2026-05-18T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,3815312.0
4,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,northamerica-northeast2,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,,1,2026-05-18T07:00:00Z,GBy.h,1,gigabyte hour,By.s,byte second,3.600000e+12,NaN,NaN,NaN,0.0,USD,0,6243470.0


In [29]:
#The cost of the SKU is units + nanos. For example, a cost of $1.75 is represented as units=1 and nanos=750,000,000. (From google documentation)
#Actions need to be made to create a new column that will contain the final price
df_final_flat['finalPrice'] = df_final_flat['unitPrice.units'].astype(float) + (df_final_flat['unitPrice.nanos'].astype(float) / 1000000000)

df_final_flat[['skuId', 'unitPrice.units', 'unitPrice.nanos', 'finalPrice']]

,skuId,unitPrice.units,unitPrice.nanos,finalPrice
0,0001-B904-8A40,0,4954950.0,0.004955
1,0001-FC8F-A9AF,0,11470000.0,0.011470
2,0006-C9C8-BB6F,0,20550000.0,0.020550
3,0007-4724-5A32,0,3815312.0,0.003815
4,0007-9388-EF75,0,6243470.0,0.006243
...,...,...,...,...
4996,27F7-3926-3BB0,0,608250.0,0.000608
4997,27FA-AB54-AC50,0,136000000.0,0.136000
4998,27FD-8FE7-7523,0,80000000.0,0.080000
4998,27FD-8FE7-7523,0,80000000.0,0.080000


In [30]:
#Just for debug to search if any line has "1" as unit price
filtered_df = df_final_flat[df_final_flat['unitPrice.units'].astype(float) == 1.0]

filtered_df[['skuId','unitPrice.units', 'unitPrice.nanos', 'finalPrice']].head()

df_final_flat.head()

,name,skuId,description,serviceRegions,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos,finalPrice
0,services/6F81-5844-456A/skus/0001-B904-8A40,0001-B904-8A40,Sole Tenancy Premium for C4 Sole Tenancy Insta...,southamerica-west1,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,southamerica-west1,,1,2026-05-18T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,4954950.0,0.004955
1,services/6F81-5844-456A/skus/0001-FC8F-A9AF,0001-FC8F-A9AF,Spot Preemptible E2 Custom Instance Core runni...,europe-west9,Google,Compute Engine,Compute,CPU,Preemptible,REGIONAL,europe-west9,,1,2026-05-18T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,11470000.0,0.011470
2,services/6F81-5844-456A/skus/0006-C9C8-BB6F,0006-C9C8-BB6F,Commitment v1: Memory-optimized Cpu in Phoenix...,us-west8,Google,Compute Engine,Compute,CPU,Commit1Yr,REGIONAL,us-west8,,1,2026-05-18T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,20550000.0,0.020550
3,services/6F81-5844-456A/skus/0007-4724-5A32,0007-4724-5A32,Sole Tenancy Premium for C4 Sole Tenancy Insta...,europe-north1,Google,Compute Engine,Compute,CPU,OnDemand,REGIONAL,europe-north1,,1,2026-05-18T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,0.0,USD,0,3815312.0,0.003815
4,services/6F81-5844-456A/skus/0007-9388-EF75,0007-9388-EF75,M4Ultramem224 Sole Tenancy Instance Ram runnin...,northamerica-northeast2,Google,Compute Engine,Compute,RAM,OnDemand,REGIONAL,northamerica-northeast2,,1,2026-05-18T07:00:00Z,GBy.h,1,gigabyte hour,By.s,byte second,3.600000e+12,NaN,NaN,NaN,0.0,USD,0,6243470.0,0.006243


In [31]:
print(df_final_flat['unitPrice.currencyCode'].value_counts())
print ()
print(df_final_flat.shape[0])
print()
print(df_final_flat['skuId'].nunique())

unitPrice.currencyCode
USD    7521
Name: count, dtype: int64

7538

5000


In [32]:
# Φιλτράρουμε τις γραμμές όπου το currencyCode δεν είναι USD
eur_rows_df = df_final_flat[df_final_flat['unitPrice.currencyCode'] != 'USD']

print(df_final_flat['unitPrice.currencyCode'].value_counts(dropna=False))

# Εμφανίζουμε τις πρώτες 5 γραμμές
eur_rows_df.head()

unitPrice.currencyCode
USD    7521
NaN      17
Name: count, dtype: int64


,name,skuId,description,serviceRegions,serviceProviderName,category.serviceDisplayName,category.resourceFamily,category.resourceGroup,category.usageType,geoTaxonomy.type,geoTaxonomy.regions,summary,currencyConversionRate,effectiveTime,pricingExpression.usageUnit,pricingExpression.displayQuantity,pricingExpression.usageUnitDescription,pricingExpression.baseUnit,pricingExpression.baseUnitDescription,pricingExpression.baseUnitConversionFactor,aggregationInfo.aggregationLevel,aggregationInfo.aggregationInterval,aggregationInfo.aggregationCount,startUsageAmount,unitPrice.currencyCode,unitPrice.units,unitPrice.nanos,finalPrice
406,services/6F81-5844-456A/skus/02FC-C3E4-9A75,02FC-C3E4-9A75,Cloud Interconnect - Data Transfer Asia Pacific,global,Google,Compute Engine,Network,PeeringOrInterconnectEgress,OnDemand,GLOBAL,NaN,Sku is not being priced by default.,0,2026-05-18T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
791,services/6F81-5844-456A/skus/05DD-028E-6CDB,05DD-028E-6CDB,Cloud Interconnect for MPS - Data Transfer Sou...,global,Google,Compute Engine,Network,PeeringOrInterconnectEgress,OnDemand,GLOBAL,NaN,Sku is not being priced by default.,0,2026-05-18T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1373,services/6F81-5844-456A/skus/0AE0-70A2-9880,0AE0-70A2-9880,Network Internet Data Transfer In from APAC to...,me-central1,Google,Compute Engine,Network,PremiumInternetIngress,OnDemand,REGIONAL,me-central1,Sku is not being priced by default.,0,2026-05-18T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1688,services/6F81-5844-456A/skus/0D5E-A385-EB21,0D5E-A385-EB21,Licensing Fee for SQL Server 2016 Standard on ...,global,Google,Compute Engine,License,SQLServer2016Standard,OnDemand,GLOBAL,NaN,Sku is not being priced by default.,0,2026-05-18T07:00:00Z,h,1,hour,s,second,3.600000e+03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2179,services/6F81-5844-456A/skus/1149-38DD-C2BE,1149-38DD-C2BE,Network Internet Data Transfer In from APAC to...,asia-northeast1,Google,Compute Engine,Network,PremiumInternetIngress,OnDemand,REGIONAL,asia-northeast1,Sku is not being priced by default.,0,2026-05-18T07:00:00Z,GiBy,1,gibibyte,By,byte,1.073742e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
#Throw duplicates, throw records with Nan in currencycode, new column with usd final price
df_clean_no_nan = df_final_flat.dropna(subset=['unitPrice.currencyCode'])

rate = df_clean_no_nan['currencyConversionRate'].astype(float)

df_clean_no_nan['final_price_usd'] = df_clean_no_nan['finalPrice'].astype(float) / rate

df_for_profiling = df_clean_no_nan.drop_duplicates(subset=['skuId'])

print(df_clean_no_nan.shape[0])
print(df_for_profiling.shape[0])

7521
4983


/tmp/ipykernel_176174/3824311183.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean_no_nan['final_price_usd'] = df_clean_no_nan['finalPrice'].astype(float) / rate


In [34]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_for_profiling, title="Google Cloud Compute Engine - Unique SKUs Report", explorative=True)

#Stores in file under the same directory
profile.to_file("google_billing_unique_analysis.html")

print("Report created")

/home/panos-varitis/anaconda3/envs/thesis/lib/python3.10/site-packages/ydata_profiling/utils/dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)
Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]/home/panos-varitis/anaconda3/envs/thesis/lib/python3.10/site-packages/ydata_profiling/visualisation/plot.py:429: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  legend = ax.legend(
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 64.88it/s]

Report created
